In [15]:
import os, re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [17]:
df = pd.read_csv('./datasets/IMDB.csv')

print(df.head())
print(df["sentiment"].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [18]:
# 모델 추론을 위한 숫자 이진 레이블 추가
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [19]:
# dataset에 포함된 HTML tag, 특수 문자 제거
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)      # HTML 태그 제거
    text = re.sub(r"[^a-zA-Z\s]", " ", text)  # 특수문자 제거
    return text.lower().strip()

# 전체 문자열에서 특수 문자 제거 후 white space를 기준으로 토큰화
def tokenize(text):
    return re.sub(r"[^a-zA-Z\s]", " ", text).lower().split()

In [20]:
sample = df["review"].iloc[0]
print("[before]\n", sample[:200])
print("\n[after]\n", clean_text(sample)[:200])

[before]
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

[after]
 one of the other reviewers has mentioned that after watching just   oz episode you ll be hooked  they are right  as this is exactly what happened with me   the first thing that struck me about oz was 


In [21]:
# pandas Series -> py List
texts  = df['review'].tolist()
labels = df['label'].tolist()

# stratify: positive/negative 비율을 train/test에 동일하게 유지
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=df['label']
)

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))  # Counter에 학습 문자열 토큰들을 누적하여 저장

In [22]:
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)  # counter에서 2번 이상 등장한 토큰을 index와 함께 저장

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab["<pad>"]
UNK_IDX    = vocab["<unk>"]
print(f"Vocab size (freq ≥ 2): {VOCAB_SIZE:,}")

# vocab에서 대응하는 정수를 통해 정수 list 반환, OOV(Out-of-Vocabulary)는 <unk>:1로 대체
def encode(text: str) -> list[int]:
    return [vocab.get(tok, UNK_IDX) for tok in tokenize(text)]

Vocab size (freq ≥ 2): 56,911


In [23]:
MAX_LEN = 64

class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.samples = []
        for text, label in zip(texts, labels):
            ids = encode(text)[:MAX_LEN]
            ids += [PAD_IDX] * (MAX_LEN - len(ids))  # 부족한 길이를 <pad>:0으로 채움
            self.samples.append((torch.tensor(ids, dtype=torch.long), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [24]:
# collate_fn 없이 기본 DataLoader 사용 (이미지 모델과 동일)
train_loader = DataLoader(IMDBDataset(train_texts, train_labels), batch_size=64, shuffle=True)
test_loader  = DataLoader(IMDBDataset(test_texts,  test_labels),  batch_size=64, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 625, Test batches: 157


In [25]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, cell_type="RNN"):
        super().__init__()

        assert cell_type in ("RNN", "GRU")
        self.cell_type = cell_type
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        rnn_cls = nn.RNN if cell_type == "RNN" else nn.GRU
        bidir   = (cell_type == "GRU")
        self.rnn = rnn_cls(
            input_size    = embed_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidir
        )
        self.dropout = nn.Dropout(dropout)

        fc_in_dim = hidden_dim * 2 if bidir else hidden_dim
        self.fc   = nn.Linear(fc_in_dim, 1)

    def forward(self, x):  # lengths 인자 없음 — 이미지 모델과 동일
        emb = self.dropout(self.embedding(x))
        _, hidden = self.rnn(emb)

        if self.cell_type == "GRU":
            h = torch.cat((hidden[-2], hidden[-1]), dim=1)  # 양방향 마지막 레이어 concat
        else:
            h = hidden[-1]  # 단방향 마지막 레이어

        return torch.sigmoid(self.fc(self.dropout(h)).squeeze(1))

In [ ]:
criterion = nn.BCELoss()

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = total_correct = total = 0

    with torch.set_grad_enabled(training):
        for x, y in loader:  # 이미지 모델과 동일한 구조
            x, y = x.to(device), y.to(device).float()
            pred = model(x)
            loss = criterion(pred, y)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss    += loss.item() * len(y)
            total_correct += ((pred >= 0.5) == y).sum().item()
            total         += len(y)

    return total_loss / total, total_correct / total

: 

In [ ]:
epoch = 10

def train_model(cell_type):
    model     = SentimentClassifier(VOCAB_SIZE, 128, 256, 2, 0.4, cell_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    history   = []

    print(f"\n[{cell_type}]  parameters: {sum(p.numel() for p in model.parameters()):,}", flush=True)
    print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Loss':>8} | {'Test Acc':>7}", flush=True)

    for e in range(1, epoch + 1):  # range(1, epoch) → 9번만 실행되는 버그 수정
        tr = run_epoch(model, train_loader, optimizer)
        vl = run_epoch(model, test_loader)
        history.append(vl)
        print(f"{e:>5} | {tr[0]:>10.4f} | {tr[1]:>9.2%} | {vl[0]:>8.4f} | {vl[1]:>7.2%}", flush=True)

    return model, history


rnn_model, rnn_hist = train_model("RNN")
gru_model, gru_hist = train_model("GRU")


[RNN]  parameters: 7,515,265
Epoch | Train Loss | Train Acc | Test Loss | Test Acc
